In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import math
import pandas as pd
import os
from os import path
from tqdm import tqdm
import json
import cv2
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import datetime
import time
import random
from PIL import Image
import pickle
import joblib
import re

from sklearn import preprocessing
import tensorflow as tf
from keras.layers import Input,Dense,LSTM,Flatten,Dropout,concatenate,Conv1D,MaxPooling2D,Activation
from keras.layers import BatchNormalization
from keras.layers import Embedding
from tensorflow.keras import initializers, regularizers
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow import keras
import tensorflow_hub as hub
from tensorflow.keras.preprocessing import image, text, sequence
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import ModelCheckpoint
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [ ]:
train_df_k1000 = pd.read_csv("/content/drive/MyDrive/VQA_Dataset/abstract_train2015_preprocessed_k999_other.csv")
train_df_k1000.tail(100)

,image_id,question_preprocessed,answer_preprocessed,answers,class_label
59900,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,where is the duck,other,"['sky and pond', 'sky', 'along pond', 'above g...",643
59901,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,are those capital or lowercase letters,other,"['capital', 'capital', 'capital', 'capital', '...",643
59902,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,are there more deer or mushrooms,other,"['no', 'same amount', 'neither', 'equal', 'sam...",643
59903,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,where is the man,other,"['beside grill', 'next to grill', 'outdoors', ...",643
59904,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,what are the little white lines on the footbal...,other,"['zones', 'laces', 'laces', 'stitches', 'hash ...",643
...,...,...,...,...,...
59995,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,what is on the rug,other,"['baby and teddy bear', 'baby and stuffed anim...",643
59996,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,what kind of fruits are on the blanket,other,"['watermelon and apples', 'watermelon', 'tomat...",643
59997,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,when will they clean the plates,other,"['after dinner', 'after they eat', 'soon', 'af...",643
59998,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,what is there about the man that might make yo...,other,"['hug', 'yes', 'gray hair', 'gray', 'smiling',...",643


In [ ]:
X = train_df_k1000[['image_id','question_preprocessed', 'answers']]
y = train_df_k1000['class_label']

In [ ]:
print(train_df_k1000.shape)
print(X.shape, y.shape)

(60000, 5)
(60000, 3) (60000,)


In [ ]:
val_df_k1000 = pd.read_csv("/content/drive/MyDrive/VQA_Dataset/abstract_val2015_preprocessed_k999_other.csv")
val_df_k1000.head(3)

,image_id,question_preprocessed,answer_preprocessed,answers,class_label
0,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,is the dog asleep,yes,"['yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'ye...",996
1,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,what is the man looking at,tv,"['tv', 'tv', 'tv', 'tv', 'tv', 'tv', 'tv', 'tv...",924
2,/content/drive/MyDrive/VQA_Dataset/scene_img_a...,is the man sitting on the armrest,yes,"['yes', 'yes', 'yes', 'yes', 'yes', 'yes', 'ye...",996


In [ ]:
X_val_test = val_df_k1000[['image_id','question_preprocessed', 'answers']]
y_val_test = val_df_k1000['class_label']

In [ ]:
print(val_df_k1000.shape)
print(X_val_test.shape, y_val_test.shape)

(29041, 5)
(29041, 3) (29041,)


## STAGE 2 - Split Train dataset into X_train and X_val  
Use Validation dataset (with class labels) as X_test to check accuracy

In [ ]:
from sklearn.model_selection import train_test_split
import pickle
from tensorflow.keras.utils import to_categorical

# Splitting into train (80%), validation (20%)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y,  random_state=42)
# Use the given validation set as test set
X_test, y_test = X_val_test, y_val_test

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

# Convert labels to categorical (before passing it to neural network model) # integer labels into one-hot encoded vectors.
Y_train = to_categorical(y_train, 1000)
Y_val = to_categorical(y_val, 1000)
Y_test = to_categorical(y_test, 1000)

print(Y_train.shape, Y_val.shape, Y_test.shape)
# Save datasets
# pickle.dump((X_train, y_train), open('/content/drive/MyDrive/VQA_Dataset/train.pkl', 'wb'))
# pickle.dump((X_val, y_val), open('/content/drive/MyDrive/VQA_Dataset/val.pkl', 'wb'))
# pickle.dump((X_test, y_test), open('/content/drive/MyDrive/VQA_Dataset/test.pkl', 'wb'))

(48000, 3) (48000,)
(12000, 3) (12000,)
(29041, 3) (29041,)
(48000, 1000) (12000, 1000) (29041, 1000)


# STAGE 3.1 - Process TRAIN and VALIDATION set for Model training

In [ ]:
# Initialize tokenizer (no default filters)
t = Tokenizer(filters='')

# Fit ONLY on training set
t.fit_on_texts(list(X_train['question_preprocessed']))  # Vocabulary built from training data only
vocab_size = len(t.word_index) + 1  # Define vocab size

# Convert text to sequences and pad them
train_sequences = t.texts_to_sequences(list(X_train['question_preprocessed']))
train_padded_docs = pad_sequences(train_sequences, maxlen=22, padding='post')

val_sequences = t.texts_to_sequences(list(X_val['question_preprocessed']))  # Use same tokenizer
val_padded_docs = pad_sequences(val_sequences, maxlen=22, padding='post')

test_sequences = t.texts_to_sequences(list(X_test['question_preprocessed']))  # Use same tokenizer
test_padded_docs = pad_sequences(test_sequences, maxlen=22, padding='post')

In [ ]:
# # Load the glove file (Using 300d version for best results)
# glove_path = "/content/drive/MyDrive/VQA_Dataset/glove.6B/glove.6B.300d.txt"

# # Load GloVe embeddings into a dictionary
# glove_vectors = {}
# with open(glove_path, "r", encoding="utf-8") as f:
#     for line in f:
#         values = line.strip().split()
#         word = values[0]  # The word token
#         vector = np.array(values[1:], dtype=np.float32)  # Convert embedding values to array
#         glove_vectors[word] = vector

# print(f"Loaded {len(glove_vectors)} word vectors.")

# # Save dictionary as pickle
# glove_pickle_path = "/content/drive/MyDrive/VQA_Dataset/glove_vectors.pkl"
# with open(glove_pickle_path, "wb") as f:
#     pickle.dump(glove_vectors, f)

# print(f"GloVe vectors saved to {glove_pickle_path} for future use.")

In [ ]:
# importing glovevector
import pickle
f = open('/content/drive/MyDrive/VQA_Dataset/glove_vectors.pkl', 'rb')
glovevector = pickle.load(f)
print('Type:',type(glovevector))
print('Size:',len(glovevector))
print('Dim:',glovevector['language'].shape)

Type: <class 'dict'>
Size: 400001
Dim: (300,)


In [ ]:
# Create embedding matrix
embedding_dim = 300  # Same as GloVe dimension
embedding_matrix = np.random.uniform(-0.05, 0.05, (vocab_size, embedding_dim))  # Random init

for word, i in t.word_index.items():
    if word in glovevector:
        embedding_matrix[i] = glovevector[word]  # Assign GloVe vector

print(embedding_matrix.shape)

(4134, 300)


In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Define augmentation layers (TensorFlow's built-in augmentation)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),  # Flip image horizontally with 50% probability
    tf.keras.layers.RandomBrightness(factor=0.2),  # Random brightness
    tf.keras.layers.RandomContrast(factor=0.2),  # Random contrast
])

def apply_augmentation(img):
    img = tf.image.resize(img, (224, 224))  # Resize for CNNs
    img = data_augmentation(img)  # Apply augmentation
    img = img / 255.0  # Normalize
    return img.numpy()  # Convert back to NumPy array for model compatibility

In [ ]:
# on training set

class CustomDataGen_aug(tf.keras.utils.Sequence):

    def __init__(self, X_que, X_img, y, batch_size, shuffle=True):
        self.X_que = X_que
        self.X_img = X_img
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(y))

    def on_epoch_end(self): # Called automatically at the end of each epoch. It shuffles the dataset to ensure the model doesn’t see data in the same order every epoch.
        if self.shuffle:
            self.indexes = np.random.permutation(self.indexes)

    def __get_input1(self, que):
        # Convert question to padded sequence
        que_arr = pad_sequences(t.texts_to_sequences([que]), maxlen=22, padding='post')[0]
        return que_arr

    def __get_input2(self, path):
        # Load image and apply augmentation
        img = cv2.imread(path)
        if img is None:
          print(f"ERROR: Image not found at {path}")
          return np.zeros((224, 224, 3))  # Return blank image if missing

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Ensure RGB format
        img = apply_augmentation(img)  # Apply augmentation & normalization
        return img

    def __get_output(self, label):
        return tf.keras.utils.to_categorical(label, num_classes=1000)

    def __getitem__(self, index):
        # Get batch data
        batch_x0 = self.X_que[index * self.batch_size:(index + 1) * self.batch_size]
        batch_x1 = self.X_img[index * self.batch_size:(index + 1) * self.batch_size]
        batch_y = self.y[index * self.batch_size:(index + 1) * self.batch_size]

        # Process data
        X0_batch = np.asarray([self.__get_input1(que) for que in batch_x0])
        X1_batch = np.asarray([self.__get_input2(path) for path in batch_x1])
        y_batch = np.asarray([self.__get_output(c) for c in batch_y])

        return (X1_batch, X0_batch), y_batch

    def __len__(self):
        return len(self.indexes) // self.batch_size

In [ ]:
# on validation split set

class CustomDataGen(tf.keras.utils.Sequence):

    # remove augmentation for validation set
    def __init__(self, X_que, X_img, y, batch_size, shuffle=True):
        self.X_que = X_que
        self.X_img = X_img
        self.y = y
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(y))

    def on_epoch_end(self):
        if self.shuffle:
            self.indexes = np.random.permutation(self.indexes)

    def __get_input1(self, que):
        que_arr = pad_sequences(t.texts_to_sequences([que]), maxlen=22, padding='post')[0]
        return que_arr

    def __get_input2(self, path):
        img = cv2.imread(path)
        if img is None:
          print(f"ERROR: Image not found at {path}")
          return np.zeros((224, 224, 3))  # Return blank image if missing
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))  # Resize to match training
        img = img / 255.0  # Normalize without augmentation
        return img

    def __get_output(self, label):
        return tf.keras.utils.to_categorical(label, num_classes=1000)

    def __getitem__(self, index):
        batch_x0 = self.X_que[index * self.batch_size:(index + 1) * self.batch_size]
        batch_x1 = self.X_img[index * self.batch_size:(index + 1) * self.batch_size]
        batch_y = self.y[index * self.batch_size:(index + 1) * self.batch_size]

        X0_batch = np.asarray([self.__get_input1(que) for que in batch_x0])
        X1_batch = np.asarray([self.__get_input2(path) for path in batch_x1])
        y_batch = np.asarray([self.__get_output(c) for c in batch_y])

        return (X1_batch, X0_batch), y_batch

    def __len__(self):
        return len(self.indexes) // self.batch_size

In [ ]:
batch_size_1 = 128
traingen = CustomDataGen_aug(list(X_train['question_preprocessed']),list(X_train['image_id']),list(y_train),batch_size=batch_size_1)
valgen = CustomDataGen(list(X_val['question_preprocessed']),list(X_val['image_id']),list(y_val),batch_size=batch_size_1)

In [ ]:
# (X_batch, y_batch) = traingen[0]  # Get first batch to confirm the shape
# print(f"Image Shape: {X_batch[0].shape}")  # Expected: (batch_size, 224, 224, 3)
# print(f"Question Shape: {X_batch[1].shape}")  # Expected: (batch_size, 22)

In [ ]:
# (X_batch, y_batch) = valgen[0]  # Get first batch to confirm the shape
# print(f"Image Shape: {X_batch[0].shape}")  # Expected: (batch_size, 224, 224, 3)
# print(f"Question Shape: {X_batch[1].shape}")  # Expected: (batch_size, 22)

# STAGE 4 - Model Training - skip to stage 5 if continuing training

In [ ]:
# import tensorflow as tf
# from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, GlobalAveragePooling2D
# from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization
# from tensorflow.keras.applications import ResNet50
# from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import Adam

# # Use Pretrained ResNet50 for Image Features
# base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
# for layer in base_model.layers:
#     layer.trainable = False  # Freeze ResNet layers

# image_input = Input(shape=(224, 224, 3))
# x = base_model(image_input, training=False)
# x = GlobalAveragePooling2D()(x)
# x = Dense(512, activation='relu')(x)  # Project image features

# # LSTM for Question Features
# question_input = Input(shape=(22,))
# embedding = Embedding(vocab_size, 300, weights=[embedding_matrix], input_length=22, trainable=False)(question_input)
# q = LSTM(128, return_sequences=True)(embedding)
# q_context = LSTM(128, return_sequences=True)(q)
# q_summary = LSTM(128)(q)  # Get summary vector

# # Cross-modal attention
# q_for_attention = Dense(512, activation='relu')(q_context)
# img_for_attention = tf.keras.layers.Reshape((1, 512))(x)  # Reshape for attention
# cross_attention = MultiHeadAttention(num_heads=8, key_dim=64)(
#     query=q_for_attention,
#     key=img_for_attention,
#     value=img_for_attention
# )
# cross_attention = LayerNormalization()(cross_attention + q_for_attention)  # Residual connection

# # Flatten and combine with question summary
# flattened = tf.keras.layers.Flatten()(cross_attention)
# combined = tf.keras.layers.concatenate([flattened, q_summary, x])

# # Fusion layers
# merged = Dense(512, activation='relu')(combined)
# merged = Dropout(0.5)(merged)
# merged = Dense(256, activation='relu')(merged)
# merged = Dropout(0.3)(merged)
# output = Dense(1000, activation='softmax')(merged)

# # Build Model
# model = Model(inputs=[image_input, question_input], outputs=output)
# model.compile(loss="categorical_crossentropy", optimizer=Adam(learning_rate=0.0001), metrics=["accuracy"])

# # Print Model Summary
# model.summary()


In [ ]:
# from tensorflow.keras.callbacks import ModelCheckpoint

# # Train model with Training and Validation sets
# batch_size = 128
# epochs = 1

# # Define checkpoint path with the correct filename extension
# checkpoint_filepath = "/content/drive/MyDrive/VQA_Dataset/model2/model_checkpoint.weights.h5"

# # Callback to save the model after each epoch
# model_checkpoint_callback = ModelCheckpoint(
#     filepath=checkpoint_filepath,
#     save_weights_only=True,  # Only save weights
#     monitor='val_loss',      # Track validation loss
#     mode='min',              # Save the model with the lowest validation loss
#     save_best_only=False,    # Save at every epoch
#     verbose=1
# )

# # Train model with checkpointing
# history = model.fit(
#     traingen,
#     validation_data=valgen,
#     epochs=epochs,
#     batch_size=batch_size,
#     callbacks=[model_checkpoint_callback],
#     verbose=1
# )


# STAGE 5 - Continue Model Training (after interruption)

### Complete stages 1 to 3.1 (till creating traingen and valgen) before this

In [ ]:
# Import dependencies
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization

# Function to Rebuild Model
def build_model():
  base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
  for layer in base_model.layers:
      layer.trainable = False  # Freeze ResNet layers

  image_input = Input(shape=(224, 224, 3))
  x = base_model(image_input, training=False)
  x = GlobalAveragePooling2D()(x)
  x = Dense(512, activation='relu')(x)  # Project image features

  # LSTM for Question Features
  question_input = Input(shape=(22,))
  embedding = Embedding(vocab_size, 300, weights=[embedding_matrix], input_length=22, trainable=False)(question_input)
  q = LSTM(128, return_sequences=True)(embedding)
  q_context = LSTM(128, return_sequences=True)(q)
  q_summary = LSTM(128)(q)  # Get summary vector

  # Cross-modal attention
  q_for_attention = Dense(512, activation='relu')(q_context)
  img_for_attention = tf.keras.layers.Reshape((1, 512))(x)  # Reshape for attention
  cross_attention = MultiHeadAttention(num_heads=8, key_dim=64)(
      query=q_for_attention,
      key=img_for_attention,
      value=img_for_attention
  )
  cross_attention = LayerNormalization()(cross_attention + q_for_attention)  # Residual connection

  # Flatten and combine with question summary
  flattened = tf.keras.layers.Flatten()(cross_attention)
  combined = tf.keras.layers.concatenate([flattened, q_summary, x])

  # Fusion layers
  merged = Dense(512, activation='relu')(combined)
  merged = Dropout(0.5)(merged)
  merged = Dense(256, activation='relu')(merged)
  merged = Dropout(0.3)(merged)
  output = Dense(1000, activation='softmax')(merged)

  # Build Model
  model = Model(inputs=[image_input, question_input], outputs=output)
  return model

# Step 1: Recreate the Model
model = build_model()

# Step 2: Load Weights from Checkpoint
checkpoint_filepath = "/content/drive/MyDrive/VQA_Dataset/final_model/model_checkpoint_epoch_22.weights.h5"
model.load_weights(checkpoint_filepath)
print("Model weights loaded successfully.")

# Step 3: Compile Model
optimizer = Adam(learning_rate=0.00001)
model.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

Model weights loaded successfully.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint_dir = "/content/drive/MyDrive/VQA_Dataset/final_model/"
every_epoch_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(checkpoint_dir, "model_checkpoint_epoch_{epoch:02d}.weights.h5"),
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=False,
    verbose=1
)

# Save only the best model weights (based on validation loss)
best_checkpoint = ModelCheckpoint(
    filepath="/content/drive/MyDrive/VQA_Dataset/final_model/best_model.weights.h5",
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,  # Save best model only
    verbose=1
)

# EarlyStopping to prevent overfitting
early_stopping_cb = EarlyStopping(
    monitor='val_loss',  # Stop if validation loss stops improving
    patience=5,  # Wait for 5 epochs before stopping
    restore_best_weights=True,  # Restore best model weights
    verbose=1
)

# Resume training with both checkpoints & early stopping
# remaining_epochs = 2 # Adjust this

history = model.fit(
    traingen,  # Training Data Generator
    validation_data=valgen,  # Validation Data Generator
    epochs=23,                # total final epoch number (ADD 2 in every round)
    initial_epoch=21,         # start from epoch 16 (ADD 2 in every round)
    # epochs=remaining_epochs,  # Remaining epochs
    # batch_size=128,  # Keep batch size the same
    callbacks=[best_checkpoint, every_epoch_checkpoint, early_stopping_cb],  # Add all callbacks
    verbose=1
)

Epoch 22/23
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 32s/step - accuracy: 0.5450 - loss: 1.5841 
Epoch 22: val_loss improved from inf to 1.72066, saving model to /content/drive/MyDrive/VQA_Dataset/final_model/best_model.weights.h5

Epoch 22: saving model to /content/drive/MyDrive/VQA_Dataset/final_model/model_checkpoint_epoch_22.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 14675s 39s/step - accuracy: 0.5450 - loss: 1.5841 - val_accuracy: 0.5145 - val_loss: 1.7207
Epoch 23/23
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 30s/step - accuracy: 0.5466 - loss: 1.5662 
Epoch 23: val_loss improved from 1.72066 to 1.71907, saving model to /content/drive/MyDrive/VQA_Dataset/final_model/best_model.weights.h5

Epoch 23: saving model to /content/drive/MyDrive/VQA_Dataset/final_model/model_checkpoint_epoch_23.weights.h5
375/375 ━━━━━━━━━━━━━━━━━━━━ 13798s 37s/step - accuracy: 0.5466 - loss: 1.5663 - val_accuracy: 0.5160 - val_loss: 1.7191
Restoring model weights from the end of the best epoch: 23.


In [ ]:
# Save the entire model after training
model.save("/content/drive/MyDrive/VQA_Dataset/final_model/full_model_23.keras")